# Notebook 05: YOLO11s-P2 + RMSA

**Model**: YOLO11s-P2 with Residual Multi-Scale Attention (RMSA) block

**RMSA Architecture**:
- Multi-scale branches (3x3 + 5x5 depthwise)
- Channel fusion via 1x1 convolution
- ECA-based channel attention
- Residual connection
- Designed for road damage (cracks, potholes, manholes)

In [ ]:
import sys
import torch
import platform

print("=" * 60)
print("ENVIRONMENT CHECK")
print("=" * 60)
print(f"Python version: {sys.version}")
print(f"PyTorch version: {torch.__version__}")

import ultralytics
print(f"Ultralytics version: {ultralytics.__version__}")

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU name: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
    DEVICE = 0
    print(f"Current device: cuda:0")
else:
    print("CUDA NOT AVAILABLE - Training will proceed on CPU")
    print(f"Reason: PyTorch build = {torch.__version__} (CPU-only build)")
    DEVICE = "cpu"
    print(f"Current device: cpu")
print(f"OS: {platform.system()} {platform.release()}")
print("=" * 60)


In [ ]:
# Shared training configuration - MUST be identical for all 5 models
import os, json, random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

TRAIN_CONFIG = {
    "data": r"D:\Nguyen-Anh-Viet\DeepLearning\DeepLearning\dataset\data.yaml",
    "imgsz": 640,
    "epochs": 100,
    "patience": 20,
    "batch": 8,
    "device": "cuda",
    "workers": 0,  # Windows safety
    "seed": SEED,
    "deterministic": True,
    "amp": True,  # AMP only with CUDA
    "pretrained": True,
    "optimizer": "auto",
    "lr0": 0.01,
    "lrf": 0.01,
    "momentum": 0.937,
    "weight_decay": 0.0005,
    "warmup_epochs": 3.0,
    "warmup_momentum": 0.8,
    "warmup_bias_lr": 0.1,
    "box": 7.5,
    "cls": 0.5,
    "dfl": 1.5,
    "hsv_h": 0.015,
    "hsv_s": 0.7,
    "hsv_v": 0.4,
    "degrees": 0.0,
    "translate": 0.1,
    "scale": 0.5,
    "shear": 0.0,
    "perspective": 0.0,
    "flipud": 0.0,
    "fliplr": 0.5,
    "mosaic": 1.0,
    "mixup": 0.0,
    "copy_paste": 0.0,
    "verbose": True,
}

RESULTS_DIR = r"D:\Nguyen-Anh-Viet\DeepLearning\DeepLearning\results"
os.makedirs(RESULTS_DIR, exist_ok=True)

print("Training configuration loaded:")
for k, v in TRAIN_CONFIG.items():
    if k != "data":
        print(f"  {k}: {v}")


In [ ]:
def benchmark_model(model_path, device, imgsz=640, warmup=20, runs=100):
    """Controlled latency benchmark for a YOLO model."""
    import time
    from ultralytics import YOLO
    
    model = YOLO(model_path)
    
    # Create dummy input
    dummy = torch.randn(1, 3, imgsz, imgsz)
    if device != "cpu":
        dummy = dummy.to(f"cuda:{device}")
    
    # Warmup
    print(f"Warming up ({warmup} iterations)...")
    for _ in range(warmup):
        _ = model.predict(source=dummy, verbose=False, device=device)
    
    # Timed runs
    print(f"Benchmarking ({runs} iterations)...")
    latencies = []
    for _ in range(runs):
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        start = time.perf_counter()
        _ = model.predict(source=dummy, verbose=False, device=device)
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        end = time.perf_counter()
        latencies.append((end - start) * 1000)  # ms
    
    mean_lat = np.mean(latencies)
    std_lat = np.std(latencies)
    fps = 1000.0 / mean_lat
    
    print(f"Latency: {mean_lat:.2f} +/- {std_lat:.2f} ms")
    print(f"FPS: {fps:.1f}")
    
    del model
    import gc; gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    
    return mean_lat, std_lat, fps


In [ ]:
def save_experiment_results(model_name, results, model_path, benchmark_results, results_dir):
    """Save experiment results to JSON for comparison notebook."""
    import os, json
    
    mean_lat, std_lat, fps = benchmark_results
    
    # Get model file size
    model_size_bytes = os.path.getsize(model_path)
    model_size_mb = model_size_bytes / (1024 * 1024)
    
    # Get model info
    from ultralytics import YOLO
    model = YOLO(model_path)
    info = model.info()
    if isinstance(info, tuple) and len(info) >= 4:
        n_layers, n_params, n_grads, gflops = info[:4]
    else:
        n_layers = len(list(model.model.modules()))
        n_params = sum(p.numel() for p in model.model.parameters())
        n_grads = sum(p.numel() for p in model.model.parameters() if p.requires_grad)
        gflops = 0.0
    
    # Extract metrics from results
    metrics = {}
    if hasattr(results, 'results_dict'):
        rd = results.results_dict
        metrics["precision"] = rd.get("metrics/precision(B)", 0)
        metrics["recall"] = rd.get("metrics/recall(B)", 0)
        metrics["mAP50"] = rd.get("metrics/mAP50(B)", 0)
        metrics["mAP50-95"] = rd.get("metrics/mAP50-95(B)", 0)
    
    # Per-class metrics if available
    per_class = {}
    if hasattr(results, 'box'):
        box = results.box
        if hasattr(box, 'ap50') and box.ap50 is not None:
            class_names = {0: "Pothole", 1: "Crack", 2: "Manhole"}
            for i, name in class_names.items():
                if i < len(box.ap50):
                    per_class[name] = {
                        "AP50": float(box.ap50[i]),
                        "AP50-95": float(box.ap[i]) if hasattr(box, 'ap') and i < len(box.ap) else 0,
                        "precision": float(box.p[i]) if hasattr(box, 'p') and i < len(box.p) else 0,
                        "recall": float(box.r[i]) if hasattr(box, 'r') and i < len(box.r) else 0,
                    }
    
    result_data = {
        "model_name": model_name,
        "model_path": model_path,
        "n_layers": int(n_layers),
        "n_params": int(n_params),
        "n_grads": int(n_grads),
        "gflops": float(gflops),
        "model_size_mb": float(model_size_mb),
        "precision": float(metrics.get("precision", 0)),
        "recall": float(metrics.get("recall", 0)),
        "mAP50": float(metrics.get("mAP50", 0)),
        "mAP50-95": float(metrics.get("mAP50-95", 0)),
        "latency_ms": float(mean_lat),
        "latency_std_ms": float(std_lat),
        "fps": float(fps),
        "per_class": per_class,
    }
    
    output_path = os.path.join(results_dir, f"{model_name}_metrics.json")
    with open(output_path, "w") as f:
        json.dump(result_data, f, indent=2)
    print(f"Results saved to {output_path}")
    
    del model
    import gc; gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    
    return result_data


## Define RMSA Module

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
import math

class ECA(nn.Module):
    """Efficient Channel Attention for use within RMSA."""
    def __init__(self, channels, gamma=2, b=1):
        super().__init__()
        t = int(abs((math.log2(channels) + b) / gamma))
        k = t if t % 2 else t + 1
        k = max(k, 3)
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.conv = nn.Conv1d(1, 1, kernel_size=k, padding=k // 2, bias=False)
        self.sigmoid = nn.Sigmoid()
    
    def forward(self, x):
        y = self.avg_pool(x)
        y = y.squeeze(-1).transpose(-1, -2)
        y = self.conv(y)
        y = y.transpose(-1, -2).unsqueeze(-1)
        y = self.sigmoid(y)
        return x * y.expand_as(x)


class RMSA(nn.Module):
    """Residual Multi-Scale Attention Block.
    
    Architecture:
    Input (C channels)
        |
        +-- Branch A: 3x3 depthwise conv -> BN -> SiLU (local features)
        |
        +-- Branch B: 5x5 depthwise conv -> BN -> SiLU (wider receptive field)
        |
        Concat (2C channels)
        |
        1x1 Conv fusion -> BN -> SiLU (reduce back to C)
        |
        ECA channel attention
        |
        1x1 Conv projection (C -> C)
        |
        + Residual (input)
        |
        Output (C channels)
    
    Uses depthwise convolutions to keep parameters low.
    """
    def __init__(self, channels):
        super().__init__()
        self.channels = channels
        
        # Branch A: 3x3 depthwise conv (local features)
        self.branch_a = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1, groups=channels, bias=False),
            nn.BatchNorm2d(channels),
            nn.SiLU(inplace=True),
        )
        
        # Branch B: 5x5 depthwise conv (wider receptive field, for cracks etc.)
        self.branch_b = nn.Sequential(
            nn.Conv2d(channels, channels, 5, padding=2, groups=channels, bias=False),
            nn.BatchNorm2d(channels),
            nn.SiLU(inplace=True),
        )
        
        # Channel fusion: 2C -> C
        self.fusion = nn.Sequential(
            nn.Conv2d(channels * 2, channels, 1, bias=False),
            nn.BatchNorm2d(channels),
            nn.SiLU(inplace=True),
        )
        
        # Channel attention
        self.eca = ECA(channels)
        
        # Projection
        self.proj = nn.Sequential(
            nn.Conv2d(channels, channels, 1, bias=False),
            nn.BatchNorm2d(channels),
        )
        
        # Final activation after residual add
        self.act = nn.SiLU(inplace=True)
    
    def forward(self, x):
        residual = x
        
        # Multi-scale branches
        a = self.branch_a(x)
        b = self.branch_b(x)
        
        # Concatenate and fuse
        out = torch.cat([a, b], dim=1)
        out = self.fusion(out)
        
        # Channel attention
        out = self.eca(out)
        
        # Projection
        out = self.proj(out)
        
        # Residual connection
        out = out + residual
        out = self.act(out)
        
        return out


# Verify RMSA
rmsa_test = RMSA(256)
test_input = torch.randn(1, 256, 32, 32)
test_output = rmsa_test(test_input)
assert test_output.shape == test_input.shape, f"Shape mismatch: {test_output.shape}"

rmsa_params = sum(p.numel() for p in rmsa_test.parameters())
print(f"RMSA module verified: {test_input.shape} -> {test_output.shape}")
print(f"RMSA parameters (for 256 channels): {rmsa_params:,}")
print(f"\nRMSA structure:")
print(rmsa_test)


## Build YOLO11s-P2 + RMSA Model

In [ ]:
from ultralytics import YOLO
from ultralytics.nn import tasks as tasks_module

MODEL_CFG = r"D:\Nguyen-Anh-Viet\DeepLearning\DeepLearning\configs\yolo11s-p2.yaml"
MODEL_NAME = "rmsa"

# Register modules
tasks_module.RMSA = RMSA
tasks_module.ECA = ECA

# Load baseline
model = YOLO(MODEL_CFG)

# Replace selected C3k2 blocks in the NECK with RMSA
# Target: P2/P3 feature processing stages (high-resolution, important for small objects)
# Layer 16: C3k2 after P3 top-down fusion
# Layer 19: C3k2 after P2 top-down fusion (most important for small damage)

target_layers = [16, 19]  # P3 and P2 fusion stages
rmsa_added = 0

for idx in target_layers:
    layer = model.model.model[idx]
    # Get output channels
    out_ch = None
    for name, mod in layer.named_modules():
        if isinstance(mod, nn.Conv2d):
            out_ch = mod.out_channels
    if out_ch is None:
        out_ch = 256
    
    # Replace C3k2 with RMSA
    rmsa_block = RMSA(out_ch)
    rmsa_block.f = getattr(layer, 'f', -1)
    rmsa_block.i = getattr(layer, 'i', -1)
    rmsa_block.type = getattr(layer, 'type', 'RMSA')
    model.model.model[idx] = rmsa_block
    rmsa_added += 1
    print(f"Replaced layer {idx} C3k2 with RMSA (channels={out_ch})")

print(f"\nTotal RMSA modules: {rmsa_added}")
assert rmsa_added > 0, "No RMSA modules added!"


## Verify RMSA Integration

In [ ]:
# Count modules
rmsa_count = sum(1 for _, m in model.model.named_modules() if isinstance(m, RMSA))
eca_in_rmsa = sum(1 for _, m in model.model.named_modules() if isinstance(m, ECA))

print(f"RMSA modules: {rmsa_count}")
print(f"ECA modules (inside RMSA): {eca_in_rmsa}")
assert rmsa_count > 0, "FAILED: No RMSA modules!"

# Parameter comparison
total_params = sum(p.numel() for p in model.model.parameters())
trainable_params = sum(p.numel() for p in model.model.parameters() if p.requires_grad)

# Load baseline for comparison
baseline = YOLO(MODEL_CFG)
baseline_params = sum(p.numel() for p in baseline.model.parameters())
param_delta = total_params - baseline_params

print(f"\nBaseline parameters: {baseline_params:,}")
print(f"RMSA model parameters: {total_params:,}")
print(f"Parameter delta: {param_delta:+,} ({param_delta/baseline_params*100:+.2f}%)")
print(f"Trainable parameters: {trainable_params:,}")

del baseline

# Forward pass
x = torch.randn(1, 3, 640, 640)
model.model.eval()
with torch.no_grad():
    output = model.model(x)
print(f"\nForward pass OK")

# Gradient verification
model.model.train()
x = torch.randn(1, 3, 640, 640)
output = model.model(x)
if isinstance(output, dict):
    loss = sum(v.sum() for v in output.values() if isinstance(v, torch.Tensor))
elif isinstance(output, (list, tuple)):
    loss = sum(o.sum() for o in output if isinstance(o, torch.Tensor))
else:
    loss = output.sum()
loss.backward()

grad_ok = False
for name, mod in model.model.named_modules():
    if isinstance(mod, RMSA):
        for pname, p in mod.named_parameters():
            if p.grad is not None:
                grad_ok = True
                print(f"RMSA gradient OK: {name}.{pname}, grad_norm={p.grad.norm().item():.6f}")
                break
        if grad_ok:
            break

assert grad_ok, "FAILED: No RMSA gradients!"
model.model.zero_grad()

# Pretrained weight report
rmsa_new_params = sum(p.numel() for _, m in model.model.named_modules() if isinstance(m, RMSA) for p in m.parameters())
pretrained_params = total_params - rmsa_new_params
print(f"\nPretrained (matched) parameters: {pretrained_params:,}")
print(f"Newly initialized RMSA parameters: {rmsa_new_params:,}")
print("\n[OK] RMSA integration fully verified")


## Train RMSA Model

In [ ]:
# Train
project_dir = os.path.join(r"D:\Nguyen-Anh-Viet\DeepLearning\DeepLearning", "runs", "rmsa")
train_config = TRAIN_CONFIG.copy()
train_config["project"] = project_dir
train_config["name"] = "train"
train_config["exist_ok"] = True

print(f"Starting RMSA model training...")

import os
best_path_check = os.path.join(project_dir, 'train', 'weights', 'best.pt')
if os.path.exists(best_path_check):
    print(f"Found {best_path_check}, skipping training!")
    results = None
else:
    results = model.train(**train_config)

print("\nTraining complete!")


## Evaluate and Benchmark

In [ ]:
best_path = os.path.join(project_dir, "train", "weights", "best.pt")
if not os.path.exists(best_path):
    import glob
    best_candidates = glob.glob(os.path.join(project_dir, "**/best.pt"), recursive=True)
    if best_candidates:
        best_path = best_candidates[0]

print(f"Best checkpoint: {best_path}")

tasks_module.RMSA = RMSA
tasks_module.ECA = ECA

model = YOLO(best_path)
val_results = model.val(data=TRAIN_CONFIG["data"], imgsz=TRAIN_CONFIG["imgsz"], device=DEVICE, workers=0)

print("\nValidation Results:")
print(f"  Precision: {val_results.results_dict['metrics/precision(B)']:.4f}")
print(f"  Recall: {val_results.results_dict['metrics/recall(B)']:.4f}")
print(f"  mAP50: {val_results.results_dict['metrics/mAP50(B)']:.4f}")
print(f"  mAP50-95: {val_results.results_dict['metrics/mAP50-95(B)']:.4f}")

benchmark_results = benchmark_model(best_path, DEVICE)
result_data = save_experiment_results(MODEL_NAME, val_results, best_path, benchmark_results, RESULTS_DIR)

print("\n" + "=" * 60)
print("RMSA RESULTS SUMMARY")
print("=" * 60)
for k, v in result_data.items():
    if k not in ["model_path", "per_class"]:
        print(f"  {k}: {v}")

del model, val_results
import gc; gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("\nDone!")
